# Advection on a finite interval: CPU implementation

This notebook solves the constant-coefficient advection equation

$$
u_t+a u_x=0, \qquad x\in[0,L], \quad a>0,
$$

with prescribed inflow at $x=0$ and a numerical closure at the outflow
boundary $x=L$. It implements upwind, Lax--Friedrichs, Lax--Wendroff,
leapfrog, FTCS, and downwind time-stepping schemes and compares four outflow closures. FTCS and downwind are retained as deliberately unstable counterexamples.

The notebook is organized as a small reusable module. The final section
contains complete examples that can be launched with a single function call.

## 1. Mathematical problem

For positive velocity $a$, characteristics travel from left to right. The
initial condition supplies the solution on characteristics starting at
$t=0$, whereas the value at $x=0$ must be prescribed for $t>0$:

$$
u(x,0)=u_0(x), \qquad u(0,t)=g(t).
$$

The examples use $g(t)=0$. The exact characteristic solution is therefore

$$
u(x,t)=
\begin{cases}
u_0(x-at), & x-at\ge 0,\\
0, & x-at<0.
\end{cases}
$$

No physical boundary condition is imposed at $x=L$, because this is an
outflow boundary. Centered finite-difference stencils nevertheless need a
numerical rule to update their last grid value; this rule is called an
*outflow closure* below.

## 2. Imports and result container

In [1]:
from __future__ import annotations

import argparse
from dataclasses import dataclass
from pathlib import Path
from typing import Sequence

import numpy as np

Array = np.ndarray

`FiniteRunResult` collects the final fields, the sampled diagnostic histories,
and the effective discretization parameters. The quantity named
`residual_energy_ratio` is retained for compatibility with the original code;
numerically it is the ratio of discrete $\ell^2$ norms, not the ratio of
their squares.
When `store_snapshots=True`, the optional arrays `snapshot_times` and `snapshots` contain the data needed to build an animation. Ordinary simulations leave them equal to `None`, avoiding unnecessary memory use.


In [2]:
@dataclass(frozen=True)
class FiniteRunResult:
    """Numerical solution and diagnostics from one finite-interval run."""

    x: Array
    numerical: Array
    exact: Array
    times: Array
    residual_history: Array
    error_history: Array
    dt: float
    nsteps: int
    courant: float
    relative_l2_error: float
    residual_energy_ratio: float
    max_residual: float
    snapshot_times: Array | None = None
    snapshots: Array | None = None


## 3. Initial data, inflow data, and exact solution

In [3]:
def compact_cosine_pulse(x: Array, center: float, half_width: float) -> Array:
    """Return a compact C1 cosine bell centered at ``center``.

    The pulse is exactly zero when ``abs(x - center) > half_width``. Compact
    support makes the residual left behind by the outflow boundary easy to
    measure after the pulse has completely left the interval.
    """
    if half_width <= 0.0:
        raise ValueError("half_width must be positive")

    distance = np.abs(x - center)
    values = np.zeros_like(x, dtype=np.float64)
    inside = distance <= half_width
    values[inside] = 0.5 * (
        1.0 + np.cos(np.pi * distance[inside] / half_width)
    )
    return values


def gaussian_pulse(x: Array, center: float, sigma: float) -> Array:
    """Return a Gaussian pulse with standard deviation ``sigma``."""
    if sigma <= 0.0:
        raise ValueError("sigma must be positive")
    return np.exp(-0.5 * ((x - center) / sigma) ** 2)


def initial_data(x: Array, case: str, length: float) -> Array:
    """Construct one of the predefined initial conditions."""
    if case == "cosine":
        return compact_cosine_pulse(
            x, center=0.25 * length, half_width=0.10 * length
        )
    if case == "gaussian":
        return gaussian_pulse(x, center=0.25 * length, sigma=0.045 * length)
    raise ValueError(f"Unknown finite-interval case: {case!r}")

In [4]:
def inflow_value(
    time: float, *, case: str, velocity: float, length: float
) -> float:
    """Return the prescribed inflow value g(t) at x=0.

    Both built-in test cases use homogeneous inflow. The otherwise unused
    arguments make this function straightforward to replace with a
    manufactured, time-dependent boundary signal.
    """
    del time, case, velocity, length
    return 0.0


def exact_finite_solution(
    x: Array,
    time: float,
    *,
    velocity: float,
    length: float,
    case: str,
) -> Array:
    """Evaluate the characteristic solution for a>0 and zero inflow."""
    if velocity <= 0.0:
        raise ValueError("This example assumes velocity > 0")
    if time < 0.0:
        raise ValueError("time must be non-negative")

    characteristic_feet = x - velocity * time
    exact = np.zeros_like(x, dtype=np.float64)
    from_initial_line = characteristic_feet >= 0.0
    if np.any(from_initial_line):
        exact[from_initial_line] = initial_data(
            characteristic_feet[from_initial_line], case, length
        )
    return exact

## 4. Numerical boundary closures

Let $u_i^n$ approximate $u(x_i,t^n)$, with $i=N_x$ at the right
boundary. After the interior values at time $n+1$ have been computed, the
implemented closures set $u_{N_x}^{n+1}$ as follows:

| Closure      | Boundary update |
|--------------|--------------|
| `frozen` | $u_{N_x}^{n+1}=u_{N_x}^n$ |
| `copy`              | $u_{N_x}^{n+1}=u_{N_x-1}^{n+1}$ |
| `extrapolation` | $u_{N_x}^{n+1}=2u_{N_x-1}^{n+1}-u_{N_x-2}^{n+1}$ |
| `outgoing` | $u_{N_x}^{n+1}=u_{N_x-1}^{n}+$ |
|            | $q(u_{N_x}^n-u_{N_x-1}^{n+1}), q=(1-C)/(1+C)$ |

Here $C=a\,\Delta t/\Delta x$ is the Courant number. The upwind method
already has a one-sided outflow stencil, so it does not call this closure.

In [5]:
def apply_outflow_closure(
    new: Array,
    old: Array,
    *,
    closure: str,
    courant: float,
) -> None:
    """Update the right endpoint after advancing the interior grid points.

    Parameters
    ----------
    new, old:
        Fields at times n+1 and n. ``new[:-1]`` must already be available.
    closure:
        One of ``frozen``, ``copy``, ``extrapolation``, or ``outgoing``.
    courant:
        Courant number C=a*dt/dx used by the discrete outgoing condition.

    Notes
    -----
    The update is performed in place on ``new[-1]``.
    """
    if closure == "frozen":
        new[-1] = old[-1]
    elif closure == "copy":
        new[-1] = new[-2]
    elif closure == "extrapolation":
        new[-1] = 2.0 * new[-2] - new[-3]
    elif closure == "outgoing":
        q = (1.0 - courant) / (1.0 + courant)
        new[-1] = old[-2] + q * (old[-1] - new[-2])
    else:
        raise ValueError(f"Unknown outflow closure: {closure!r}")

## 5. Time-stepping schemes

The spatial grid is uniform, $x_j=j\Delta x$. Upwind,
Lax--Friedrichs, Lax--Wendroff, FTCS, and downwind are one-step methods.
Leapfrog is a two-step method and therefore obtains its first level from a
selected startup scheme.

For positive velocity, the stable schemes require $0<C\le 1$. Upwind and
Lax--Friedrichs are first-order accurate and numerically dissipative. It is
important not to interpret Lax--Friedrichs as a cure for upwind diffusion: for
$0<C<1$, its leading artificial-diffusion coefficient is larger than that of
upwind. Lax--Wendroff and leapfrog are second-order methods and are much less
dissipative, although dispersive oscillations can occur near poorly resolved
gradients.

FTCS and downwind are included only as instructive counterexamples. FTCS is
unconditionally unstable for pure advection. Downwind uses information from
the wrong side of the characteristic and is also unstable for $a>0$.


In [6]:
def one_step_finite(
    old: Array,
    *,
    method: str,
    closure: str,
    courant: float,
    inflow: float,
) -> Array:
    """Advance a one-step finite-difference method by one time level."""
    new = np.empty_like(old)

    # Positive velocity makes x=0 the inflow boundary.
    new[0] = inflow

    if method == "upwind":
        # This one-sided stencil uses only the current point and its left
        # neighbor. It therefore advances the outflow point without a ghost
        # value or a separate numerical closure.
        new[1:] = old[1:] - courant * (old[1:] - old[:-1])
        return new

    # Common slices for centered three-point stencils.
    right = old[2:]
    center = old[1:-1]
    left = old[:-2]

    if method == "lax-friedrichs":
        new[1:-1] = 0.5 * (right + left) - 0.5 * courant * (right - left)
    elif method == "downwind":
        # Deliberately incorrect for a>0: the forward spatial difference
        # looks away from the upwind characteristic. Its amplification
        # factor exceeds one for some Fourier modes.
        new[1:-1] = center - courant * (right - center)
    elif method == "lax-wendroff":
        new[1:-1] = (
            center
            - 0.5 * courant * (right - left)
            + 0.5 * courant**2 * (right - 2.0 * center + left)
        )
    elif method == "ftcs":
        new[1:-1] = center - 0.5 * courant * (right - left)
    else:
        raise ValueError(f"Unsupported one-step method: {method!r}")

    apply_outflow_closure(new, old, closure=closure, courant=courant)
    return new


def leapfrog_start(
    initial: Array,
    *,
    startup: str,
    closure: str,
    courant: float,
    inflow: float,
) -> Array:
    """Construct U^1 for leapfrog using a selected one-step method."""
    if startup not in {"lax-wendroff", "upwind", "ftcs"}:
        raise ValueError(f"Unknown leapfrog startup: {startup!r}")
    return one_step_finite(
        initial,
        method=startup,
        closure=closure,
        courant=courant,
        inflow=inflow,
    )

## 6. Solver

In [7]:
def solve_finite_interval(
    method: str,
    *,
    closure: str = "outgoing",
    nx: int = 400,
    velocity: float = 1.0,
    cfl: float = 0.8,
    final_time: float = 1.10,
    length: float = 1.0,
    case: str = "cosine",
    startup: str = "lax-wendroff",
    save_every: int = 1,
    store_snapshots: bool = False,
    snapshot_every: int = 1,
) -> FiniteRunResult:
    """Solve the finite-interval advection problem for positive velocity.

    ``nx`` is the number of cells, so the returned grid contains ``nx+1``
    points. The requested CFL value first defines a tentative time step. The
    number of steps is then rounded up and ``dt`` is adjusted so that the last
    step lands exactly on ``final_time``; consequently the effective Courant
    number can be slightly smaller than ``cfl``.

    Errors and residuals are normalized by the initial discrete l2 norm. The
    histories are stored every ``save_every`` steps and always at final time.
    Set ``store_snapshots=True`` to retain full solution fields for an
    animation; ``snapshot_every`` controls their temporal stride.
    """
    supported_methods = {
        "upwind", "downwind", "lax-friedrichs", "lax-wendroff",
        "leapfrog", "ftcs"
    }
    supported_closures = {"frozen", "copy", "extrapolation", "outgoing"}
    if method not in supported_methods:
        raise ValueError(f"Unknown method: {method!r}")
    if closure not in supported_closures:
        raise ValueError(f"Unknown outflow closure: {closure!r}")
    if nx < 8:
        raise ValueError("nx must be at least 8")
    if velocity <= 0.0:
        raise ValueError("This implementation assumes velocity > 0")
    if length <= 0.0:
        raise ValueError("length must be positive")
    if final_time < 0.0:
        raise ValueError("final_time must be non-negative")
    if not (0.0 < cfl <= 1.0):
        raise ValueError("cfl must satisfy 0 < cfl <= 1")
    if save_every < 1:
        raise ValueError("save_every must be positive")
    if snapshot_every < 1:
        raise ValueError("snapshot_every must be positive")

    dx = length / nx
    tentative_dt = cfl * dx / velocity
    nsteps = max(1, int(np.ceil(final_time / tentative_dt)))
    dt = final_time / nsteps
    courant = velocity * dt / dx
    x = np.linspace(0.0, length, nx + 1, dtype=np.float64)

    initial = initial_data(x, case, length)
    initial_norm = max(float(np.linalg.norm(initial)), 1.0e-30)
    residual_history = [1.0]
    error_history = [0.0]
    times = [0.0]
    snapshot_times = [0.0] if store_snapshots else []
    snapshots = [initial.copy()] if store_snapshots else []

    def record(field: Array, step: int) -> None:
        """Append diagnostics for one accepted time level."""
        time_now = step * dt
        exact_now = exact_finite_solution(
            x, time_now, velocity=velocity, length=length, case=case
        )
        residual_history.append(float(np.linalg.norm(field) / initial_norm))
        error_history.append(float(np.linalg.norm(field - exact_now) / initial_norm))
        times.append(time_now)

    def record_snapshot(field: Array, step: int) -> None:
        """Store an independent field copy for later animation."""
        if store_snapshots and (
            step % snapshot_every == 0 or step == nsteps
        ):
            snapshot_times.append(step * dt)
            snapshots.append(field.copy())

    if method == "leapfrog":
        previous = initial.copy()  # U^0
        current = leapfrog_start(  # U^1
            previous,
            startup=startup,
            closure=closure,
            courant=courant,
            inflow=inflow_value(
                dt, case=case, velocity=velocity, length=length
            ),
        )
        if 1 % save_every == 0 or nsteps == 1:
            record(current, 1)
        record_snapshot(current, 1)

        for step in range(1, nsteps):
            next_field = np.empty_like(current)
            next_field[0] = inflow_value(
                (step + 1) * dt,
                case=case,
                velocity=velocity,
                length=length,
            )
            next_field[1:-1] = previous[1:-1] - courant * (
                current[2:] - current[:-2]
            )
            apply_outflow_closure(
                next_field, current, closure=closure, courant=courant
            )
            previous, current = current, next_field
            if (step + 1) % save_every == 0 or step + 1 == nsteps:
                record(current, step + 1)
            record_snapshot(current, step + 1)
        numerical = current
    else:
        numerical = initial.copy()
        for step in range(nsteps):
            numerical = one_step_finite(
                numerical,
                method=method,
                closure=closure,
                courant=courant,
                inflow=inflow_value(
                    (step + 1) * dt,
                    case=case,
                    velocity=velocity,
                    length=length,
                ),
            )
            if (step + 1) % save_every == 0 or step + 1 == nsteps:
                record(numerical, step + 1)
            record_snapshot(numerical, step + 1)

    exact = exact_finite_solution(
        x, final_time, velocity=velocity, length=length, case=case
    )
    relative_l2 = float(np.linalg.norm(numerical - exact) / initial_norm)
    residual_ratio = float(np.linalg.norm(numerical) / initial_norm)

    return FiniteRunResult(
        x=x,
        numerical=numerical,
        exact=exact,
        times=np.asarray(times, dtype=np.float64),
        residual_history=np.asarray(residual_history, dtype=np.float64),
        error_history=np.asarray(error_history, dtype=np.float64),
        dt=dt,
        nsteps=nsteps,
        courant=courant,
        relative_l2_error=relative_l2,
        residual_energy_ratio=residual_ratio,
        max_residual=float(np.max(np.abs(numerical))),
        snapshot_times=(
            np.asarray(snapshot_times, dtype=np.float64)
            if store_snapshots else None
        ),
        snapshots=(np.stack(snapshots) if store_snapshots else None),
    )

## 7. Reporting and plotting helpers

In [8]:
def print_run_summary(
    result: FiniteRunResult,
    *,
    method: str,
    closure: str,
    case: str,
    nx: int,
) -> None:
    """Print a compact, consistently formatted summary of one run."""
    print(f"method                  = {method}")
    print(f"outflow closure         = {closure}")
    print(f"case                    = {case}")
    print(f"grid cells              = {nx}")
    print(f"time steps              = {result.nsteps}")
    print(f"dt                      = {result.dt:.12e}")
    print(f"Courant number          = {result.courant:.12e}")
    print(f"relative L2 error       = {result.relative_l2_error:.12e}")
    print(f"residual L2 ratio       = {result.residual_energy_ratio:.12e}")
    print(f"maximum residual        = {result.max_residual:.12e}")


def plot_final_solution(
    result: FiniteRunResult,
    *,
    label: str = "Numerical",
    title: str | None = None,
    output: Path | str | None = None,
    show: bool = True,
):
    """Plot the final numerical and exact solutions and return the figure."""
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(7.2, 4.5))
    ax.plot(result.x, result.exact, linewidth=2.2, label="Exact")
    ax.plot(result.x, result.numerical, linewidth=1.5, label=label)
    ax.set_xlabel("x")
    ax.set_ylabel("u")
    if title is not None:
        ax.set_title(title)
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()

    if output is not None:
        output = Path(output)
        output.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(output, dpi=180)
    if show:
        plt.show()
    return fig

## 8. Complete example functions

The following functions are the recommended notebook interface:

- `run_single_example` runs, reports, and plots one simulation;
- `run_closure_example` compares the four outflow closures;
- `run_method_comparison_example` compares the stable time integrators before
  the pulse reaches the outflow boundary.

Each function returns its numerical results. Set `show=False` for automated or
non-interactive execution, and pass `output=...` to save the corresponding
figure.

In [9]:
def run_single_example(
    *,
    method: str = "lax-wendroff",
    closure: str = "outgoing",
    case: str = "cosine",
    nx: int = 400,
    velocity: float = 1.0,
    cfl: float = 0.8,
    final_time: float = 1.10,
    length: float = 1.0,
    startup: str = "lax-wendroff",
    save_every: int = 1,
    output: Path | str | None = None,
    show: bool = True,
) -> FiniteRunResult:
    """Run one complete simulation, print diagnostics, and make a plot."""
    result = solve_finite_interval(
        method,
        closure=closure,
        case=case,
        nx=nx,
        velocity=velocity,
        cfl=cfl,
        final_time=final_time,
        length=length,
        startup=startup,
        save_every=save_every,
    )
    print_run_summary(
        result, method=method, closure=closure, case=case, nx=nx
    )
    plot_final_solution(
        result,
        label=f"{method}, {closure}",
        title=f"Advection at t={final_time:g}",
        output=output,
        show=show,
    )
    return result


def closure_study(
    *,
    nx: int = 400,
    velocity: float = 1.0,
    cfl: float = 0.8,
    final_time: float = 1.10,
    length: float = 1.0,
    method: str = "lax-wendroff",
    case: str = "cosine",
    save_every: int = 1,
) -> dict[str, FiniteRunResult]:
    """Run the same problem with all four outflow closures."""
    return {
        closure: solve_finite_interval(
            method,
            closure=closure,
            nx=nx,
            velocity=velocity,
            cfl=cfl,
            final_time=final_time,
            length=length,
            case=case,
            save_every=save_every,
        )
        for closure in ("frozen", "copy", "extrapolation", "outgoing")
    }


def run_closure_example(
    *,
    nx: int = 160,
    velocity: float = 1.0,
    cfl: float = 0.8,
    crossing_time: float = 0.75,
    history_time: float = 0.90,
    length: float = 1.0,
    method: str = "lax-wendroff",
    case: str = "cosine",
    output: Path | str | None = None,
    show: bool = True,
) -> dict[str, FiniteRunResult]:
    """Compare outflow closures while and after the pulse crosses x=L."""
    import matplotlib.pyplot as plt

    crossing = closure_study(
        nx=nx,
        velocity=velocity,
        cfl=cfl,
        final_time=crossing_time,
        length=length,
        method=method,
        case=case,
    )
    histories = closure_study(
        nx=nx,
        velocity=velocity,
        cfl=cfl,
        final_time=history_time,
        length=length,
        method=method,
        case=case,
    )

    print("Outflow-closure study at final history time:")
    for closure, result in histories.items():
        print(
            f"  {closure:13s}: residual L2 ratio = "
            f"{result.residual_energy_ratio:.6e}, "
            f"max residual = {result.max_residual:.6e}"
        )

    fig, axes = plt.subplots(2, 1, figsize=(8.0, 7.2))
    reference = crossing["outgoing"]
    axes[0].plot(reference.x, reference.exact, linewidth=2.2, label="Exact")
    for closure, result in crossing.items():
        axes[0].plot(result.x, result.numerical, label=closure.capitalize())
    axes[0].set_title("Pulse crossing the numerical outflow boundary")
    axes[0].set_ylabel("u")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend(ncol=3, fontsize=8)

    for closure, result in histories.items():
        axes[1].semilogy(
            result.times,
            np.maximum(result.error_history, 1.0e-16),
            label=closure.capitalize(),
        )
    pulse_reaches_outflow = (
        length - (0.25 * length + 0.10 * length)
    ) / velocity
    axes[1].axvline(
        pulse_reaches_outflow,
        linestyle="--",
        linewidth=1.0,
        label="Pulse reaches outflow",
    )
    axes[1].set_xlabel("Time")
    axes[1].set_ylabel(r"$\|u_h-u\|_2/\|u_h(0)\|_2$")
    axes[1].grid(True, which="both", alpha=0.3)
    axes[1].legend(ncol=3, fontsize=8)
    fig.tight_layout()

    if output is not None:
        output = Path(output)
        output.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(output, dpi=180)
    if show:
        plt.show()
    return histories


def run_method_comparison_example(
    *,
    methods: Sequence[str] = (
        "upwind", "lax-friedrichs", "lax-wendroff", "leapfrog"
    ),
    nx: int = 200,
    velocity: float = 1.0,
    cfl: float = 0.8,
    final_time: float = 0.40,
    length: float = 1.0,
    case: str = "cosine",
    closure: str = "outgoing",
    output: Path | str | None = None,
    show: bool = True,
) -> dict[str, FiniteRunResult]:
    """Compare stable schemes before boundary effects dominate the error."""
    import matplotlib.pyplot as plt

    results = {
        method: solve_finite_interval(
            method,
            closure=closure,
            nx=nx,
            velocity=velocity,
            cfl=cfl,
            final_time=final_time,
            length=length,
            case=case,
        )
        for method in methods
    }

    print("Method comparison:")
    for method, result in results.items():
        print(f"  {method:15s}: relative L2 error = {result.relative_l2_error:.6e}")

    fig, ax = plt.subplots(figsize=(8.0, 4.8))
    reference = next(iter(results.values()))
    ax.plot(reference.x, reference.exact, "k--", linewidth=2.2, label="Exact")
    for method, result in results.items():
        ax.plot(result.x, result.numerical, label=method)
    ax.set_title(f"Method comparison at t={final_time:g}")
    ax.set_xlabel("x")
    ax.set_ylabel("u")
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()

    if output is not None:
        output = Path(output)
        output.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(output, dpi=180)
    if show:
        plt.show()
    return results


def generate_boundary_figure(output: Path | str) -> dict[str, FiniteRunResult]:
    """Backward-compatible wrapper that saves the closure-study figure."""
    return run_closure_example(output=output, show=False)

## 9. Animation and video download

`run_animation_example` carries out a complete animated experiment:

1. it chooses a snapshot stride that limits the number of video frames;
2. it runs the same finite-difference solver used by the static examples;
3. it creates an MP4 animation with the numerical and exact solutions;
4. it optionally embeds the video in the notebook;
5. it optionally starts the browser download.

MP4 encoding uses `ffmpeg`. Google Colab normally includes it; if it is absent,
`ensure_ffmpeg` installs it automatically in the Colab runtime. The generated
file is placed in the current working directory (normally `/content` in Colab).

The animation functions do not depend on Colab for the numerical calculation
or video generation. Only the download step is environment-specific:
`google.colab.files.download` is used in Colab, while a clickable `FileLink` is
displayed in a standard Jupyter notebook.

In [10]:
def _running_in_colab() -> bool:
    """Return True when the current kernel is hosted by Google Colab."""
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


def ensure_ffmpeg() -> None:
    """Ensure that Matplotlib can access an ffmpeg MP4 writer.

    Colab usually ships with ffmpeg. If a future runtime does not include it,
    this function installs the package through ``apt-get``. Outside Colab it
    raises an actionable error instead of modifying the local environment.
    """
    import shutil
    import subprocess

    if shutil.which("ffmpeg") is not None:
        return
    if not _running_in_colab():
        raise RuntimeError(
            "ffmpeg was not found. Install ffmpeg or run the notebook in "
            "Google Colab before requesting an MP4 file."
        )

    print("ffmpeg not found; installing it in the Colab runtime...")
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=True)
    if shutil.which("ffmpeg") is None:
        raise RuntimeError("The ffmpeg installation did not complete successfully")


def download_generated_file(path: Path | str) -> Path:
    """Download a generated file in Colab or display a Jupyter file link."""
    path = Path(path).resolve()
    if not path.is_file():
        raise FileNotFoundError(path)

    if _running_in_colab():
        from google.colab import files

        files.download(str(path))
    else:
        from IPython.display import FileLink, display as ipy_display

        print("Outside Google Colab: use the link below to download the video.")
        ipy_display(FileLink(str(path)))
    return path


def create_solution_animation(
    result: FiniteRunResult,
    *,
    velocity: float,
    length: float,
    case: str,
    method: str,
    closure: str,
    output: Path | str = "advection_propagation.mp4",
    fps: int = 25,
    dpi: int = 120,
    preview: bool = True,
    dynamic_ylim: bool = False,
) -> Path:
    """Create, save, and optionally display an MP4 solution animation.

    The numerical curve is reconstructed from the snapshots stored in
    ``result``. The exact curve is evaluated at the same snapshot times, so
    phase and amplitude errors remain visible throughout the propagation.
    """
    import matplotlib.pyplot as plt
    from matplotlib.animation import FFMpegWriter, FuncAnimation

    if result.snapshots is None or result.snapshot_times is None:
        raise ValueError(
            "No snapshots are available. Run solve_finite_interval with "
            "store_snapshots=True."
        )
    if fps < 1:
        raise ValueError("fps must be positive")
    if dpi < 40:
        raise ValueError("dpi must be at least 40")

    ensure_ffmpeg()
    output = Path(output)
    if output.suffix.lower() != ".mp4":
        raise ValueError("The animation output filename must end in .mp4")
    output.parent.mkdir(parents=True, exist_ok=True)

    snapshots = result.snapshots
    snapshot_times = result.snapshot_times
    data_min = min(float(np.min(snapshots)), 0.0)
    data_max = max(float(np.max(snapshots)), 1.0e-12)
    padding = max(0.08 * (data_max - data_min), 0.05)

    fig, ax = plt.subplots(figsize=(8.0, 4.8))
    numerical_line, = ax.plot([], [], linewidth=2.0, label="Numerical")
    exact_line, = ax.plot([], [], "k--", linewidth=1.8, label="Exact")
    time_text = ax.text(
        0.02, 0.96, "", transform=ax.transAxes, va="top"
    )
    diagnostic_text = ax.text(
        0.02, 0.88, "", transform=ax.transAxes, va="top"
    )
    initial_norm = max(float(np.linalg.norm(snapshots[0])), 1.0e-30)
    ax.set_xlim(float(result.x[0]), float(result.x[-1]))
    ax.set_ylim(data_min - padding, data_max + padding)
    ax.set_xlabel("x")
    ax.set_ylabel("u")
    ax.set_title(f"{method}; outflow closure: {closure}; C={result.courant:.3f}")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper right")
    fig.tight_layout()

    def initialize():
        numerical_line.set_data([], [])
        exact_line.set_data([], [])
        time_text.set_text("")
        diagnostic_text.set_text("")
        return numerical_line, exact_line, time_text, diagnostic_text

    def update(frame: int):
        time_now = float(snapshot_times[frame])
        exact_now = exact_finite_solution(
            result.x,
            time_now,
            velocity=velocity,
            length=length,
            case=case,
        )
        numerical_now = snapshots[frame]
        numerical_line.set_data(result.x, numerical_now)
        exact_line.set_data(result.x, exact_now)
        relative_error = float(
            np.linalg.norm(numerical_now - exact_now) / initial_norm
        )
        maximum = float(np.max(np.abs(numerical_now)))
        time_text.set_text(f"t = {time_now:.4f}")
        diagnostic_text.set_text(
            f"relative L2 error = {relative_error:.3e}\n"
            f"max |u_h| = {maximum:.3e}"
        )
        if dynamic_ylim:
            frame_min = min(float(np.min(numerical_now)), 0.0)
            frame_max = max(float(np.max(numerical_now)), 1.0)
            frame_padding = max(0.08 * (frame_max - frame_min), 0.05)
            ax.set_ylim(
                frame_min - frame_padding, frame_max + frame_padding
            )
        return numerical_line, exact_line, time_text, diagnostic_text

    animation = FuncAnimation(
        fig,
        update,
        frames=len(snapshot_times),
        init_func=initialize,
        interval=1000.0 / fps,
        blit=not dynamic_ylim,
    )
    writer = FFMpegWriter(
        fps=fps,
        bitrate=1800,
        metadata={"title": "Finite-interval advection"},
    )
    animation.save(str(output), writer=writer, dpi=dpi)
    plt.close(fig)

    print(
        f"Video saved to {output.resolve()} "
        f"({len(snapshot_times)} frames, {fps} fps)."
    )
    if preview:
        from IPython.display import Video, display as ipy_display

        ipy_display(Video(str(output), embed=True))
    return output.resolve()


def run_animation_example(
    *,
    method: str = "lax-wendroff",
    closure: str = "outgoing",
    case: str = "cosine",
    nx: int = 400,
    velocity: float = 1.0,
    cfl: float = 0.8,
    final_time: float = 1.10,
    length: float = 1.0,
    startup: str = "lax-wendroff",
    max_frames: int = 180,
    fps: int = 25,
    dpi: int = 120,
    output: Path | str = "advection_propagation.mp4",
    preview: bool = True,
    download: bool = False,
    dynamic_ylim: bool = False,
) -> tuple[FiniteRunResult, Path]:
    """Run a simulation and produce a downloadable propagation video.

    ``max_frames`` limits file size and encoding time without changing the
    numerical time step: only the frequency at which fields are copied for the
    animation is changed. Set ``download=True`` to start the Colab download as
    soon as encoding is complete.
    """
    if max_frames < 2:
        raise ValueError("max_frames must be at least 2")

    dx = length / nx
    tentative_dt = cfl * dx / velocity
    estimated_steps = max(1, int(np.ceil(final_time / tentative_dt)))
    snapshot_every = max(1, int(np.ceil(estimated_steps / (max_frames - 1))))

    result = solve_finite_interval(
        method,
        closure=closure,
        case=case,
        nx=nx,
        velocity=velocity,
        cfl=cfl,
        final_time=final_time,
        length=length,
        startup=startup,
        save_every=max(1, snapshot_every),
        store_snapshots=True,
        snapshot_every=snapshot_every,
    )
    print_run_summary(
        result, method=method, closure=closure, case=case, nx=nx
    )
    video_path = create_solution_animation(
        result,
        velocity=velocity,
        length=length,
        case=case,
        method=method,
        closure=closure,
        output=output,
        fps=fps,
        dpi=dpi,
        preview=preview,
        dynamic_ylim=dynamic_ylim,
    )
    if download:
        download_generated_file(video_path)
    return result, video_path


def create_closure_comparison_animation(
    results: dict[str, FiniteRunResult],
    *,
    velocity: float,
    length: float,
    case: str,
    method: str = "leapfrog",
    output: Path | str = "leapfrog_closure_comparison.mp4",
    fps: int = 25,
    dpi: int = 120,
    preview: bool = True,
) -> Path:
    """Animate several outflow closures and their relative-error histories."""
    import matplotlib.pyplot as plt
    from matplotlib.animation import FFMpegWriter, FuncAnimation

    if not results:
        raise ValueError("At least one closure result is required")
    ensure_ffmpeg()
    output = Path(output)
    if output.suffix.lower() != ".mp4":
        raise ValueError("The animation output filename must end in .mp4")
    output.parent.mkdir(parents=True, exist_ok=True)

    reference = next(iter(results.values()))
    if reference.snapshot_times is None or reference.snapshots is None:
        raise ValueError("Closure results must contain stored snapshots")
    snapshot_times = reference.snapshot_times
    initial_norm = max(float(np.linalg.norm(reference.snapshots[0])), 1.0e-30)

    for closure, result in results.items():
        if result.snapshot_times is None or result.snapshots is None:
            raise ValueError(f"No snapshots are available for {closure!r}")
        if not np.array_equal(result.snapshot_times, snapshot_times):
            raise ValueError("Closure runs must use identical snapshot times")

    exact_snapshots = np.stack([
        exact_finite_solution(
            reference.x,
            float(time_now),
            velocity=velocity,
            length=length,
            case=case,
        )
        for time_now in snapshot_times
    ])
    error_histories = {
        closure: np.linalg.norm(result.snapshots - exact_snapshots, axis=1)
        / initial_norm
        for closure, result in results.items()
    }

    all_values = np.concatenate([result.snapshots.ravel() for result in results.values()])
    data_min = min(float(np.min(all_values)), 0.0)
    data_max = max(float(np.max(all_values)), 1.0)
    padding = max(0.08 * (data_max - data_min), 0.05)
    positive_errors = np.concatenate([
        np.maximum(errors, 1.0e-15) for errors in error_histories.values()
    ])
    error_max = max(float(np.max(positive_errors)), 1.0e-12)

    fig, axes = plt.subplots(
        2, 1, figsize=(8.4, 7.0), gridspec_kw={"height_ratios": [2.0, 1.0]}
    )
    exact_line, = axes[0].plot([], [], "k--", linewidth=2.0, label="Exact")
    solution_lines = {
        closure: axes[0].plot([], [], linewidth=1.5, label=closure.capitalize())[0]
        for closure in results
    }
    time_text = axes[0].text(0.02, 0.94, "", transform=axes[0].transAxes)
    axes[0].set_xlim(float(reference.x[0]), float(reference.x[-1]))
    axes[0].set_ylim(data_min - padding, data_max + padding)
    axes[0].set_ylabel("u")
    axes[0].set_title("Leapfrog: comparison of outflow closures")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend(ncol=3, fontsize=8)

    error_lines = {
        closure: axes[1].semilogy([], [], linewidth=1.5, label=closure.capitalize())[0]
        for closure in results
    }
    axes[1].set_xlim(float(snapshot_times[0]), float(snapshot_times[-1]))
    axes[1].set_ylim(1.0e-15, 1.5 * error_max)
    axes[1].set_xlabel("Time")
    axes[1].set_ylabel(r"$\|u_h-u\|_2/\|u_h(0)\|_2$")
    axes[1].grid(True, which="both", alpha=0.3)
    axes[1].legend(ncol=2, fontsize=8)
    fig.tight_layout()

    def initialize():
        exact_line.set_data([], [])
        time_text.set_text("")
        artists = [exact_line, time_text]
        for line in solution_lines.values():
            line.set_data([], [])
            artists.append(line)
        for line in error_lines.values():
            line.set_data([], [])
            artists.append(line)
        return tuple(artists)

    def update(frame: int):
        exact_line.set_data(reference.x, exact_snapshots[frame])
        time_text.set_text(f"t = {snapshot_times[frame]:.4f}")
        artists = [exact_line, time_text]
        for closure, result in results.items():
            solution_lines[closure].set_data(reference.x, result.snapshots[frame])
            error_lines[closure].set_data(
                snapshot_times[: frame + 1],
                np.maximum(error_histories[closure][: frame + 1], 1.0e-15),
            )
            artists.extend([solution_lines[closure], error_lines[closure]])
        return tuple(artists)

    animation = FuncAnimation(
        fig,
        update,
        frames=len(snapshot_times),
        init_func=initialize,
        interval=1000.0 / fps,
        blit=True,
    )
    writer = FFMpegWriter(
        fps=fps,
        bitrate=2200,
        metadata={"title": "Leapfrog outflow-closure comparison"},
    )
    animation.save(str(output), writer=writer, dpi=dpi)
    plt.close(fig)

    print(
        f"Video saved to {output.resolve()} "
        f"({len(snapshot_times)} frames, {fps} fps)."
    )
    if preview:
        from IPython.display import Video, display as ipy_display

        ipy_display(Video(str(output), embed=True))
    return output.resolve()


def run_closure_animation_example(
    *,
    closures: Sequence[str] = ("frozen", "copy", "extrapolation", "outgoing"),
    nx: int = 120,
    velocity: float = 1.0,
    cfl: float = 0.8,
    final_time: float = 1.10,
    length: float = 1.0,
    case: str = "cosine",
    startup: str = "lax-wendroff",
    max_frames: int = 120,
    fps: int = 20,
    dpi: int = 110,
    output: Path | str = "05_leapfrog_closure_comparison.mp4",
    preview: bool = True,
    download: bool = False,
) -> tuple[dict[str, FiniteRunResult], Path]:
    """Run Leapfrog with several closures and create one comparison video."""
    if max_frames < 2:
        raise ValueError("max_frames must be at least 2")
    dx = length / nx
    tentative_dt = cfl * dx / velocity
    estimated_steps = max(1, int(np.ceil(final_time / tentative_dt)))
    snapshot_every = max(1, int(np.ceil(estimated_steps / (max_frames - 1))))

    results = {
        closure: solve_finite_interval(
            "leapfrog",
            closure=closure,
            nx=nx,
            velocity=velocity,
            cfl=cfl,
            final_time=final_time,
            length=length,
            case=case,
            startup=startup,
            save_every=snapshot_every,
            store_snapshots=True,
            snapshot_every=snapshot_every,
        )
        for closure in closures
    }
    video_path = create_closure_comparison_animation(
        results,
        velocity=velocity,
        length=length,
        case=case,
        output=output,
        fps=fps,
        dpi=dpi,
        preview=preview,
    )
    if download:
        download_generated_file(video_path)
    return results, video_path




def _interpolate_snapshot_history(
    result: FiniteRunResult, target_times: Array
) -> Array:
    """Linearly interpolate stored fields onto common animation times.

    This interpolation is used only to synchronize video frames from runs with
    different time steps. It does not alter either numerical simulation or any
    error reported at the actual computed time levels.
    """
    if result.snapshots is None or result.snapshot_times is None:
        raise ValueError("The result does not contain animation snapshots")
    times = result.snapshot_times
    if times.size < 2:
        raise ValueError("At least two snapshots are required")
    if target_times[0] < times[0] or target_times[-1] > times[-1]:
        raise ValueError("Target animation times lie outside the stored history")

    right = np.searchsorted(times, target_times, side="right")
    right = np.clip(right, 1, times.size - 1)
    left = right - 1
    denominator = times[right] - times[left]
    weight = (target_times - times[left]) / denominator
    return (
        (1.0 - weight)[:, None] * result.snapshots[left]
        + weight[:, None] * result.snapshots[right]
    )


def _select_synchronized_frame_times(
    baseline: FiniteRunResult,
    mitigated: FiniteRunResult,
    max_frames: int,
) -> Array:
    """Prefer time levels that were computed by both simulations.

    Common computed levels avoid introducing a visual interpolation error. If
    two arbitrary user configurations have too few common levels, the function
    falls back to uniformly spaced physical times and the histories are then
    interpolated only for display.
    """
    baseline_times = baseline.snapshot_times
    mitigated_times = mitigated.snapshot_times
    if baseline_times is None or mitigated_times is None:
        raise ValueError("Both simulations must contain snapshot times")

    tolerance = 100.0 * np.finfo(float).eps * max(
        1.0, float(baseline_times[-1]), float(mitigated_times[-1])
    )
    common = []
    left_index = 0
    right_index = 0
    while left_index < len(baseline_times) and right_index < len(mitigated_times):
        difference = baseline_times[left_index] - mitigated_times[right_index]
        if abs(difference) <= tolerance:
            common.append(float(baseline_times[left_index]))
            left_index += 1
            right_index += 1
        elif difference < 0.0:
            left_index += 1
        else:
            right_index += 1

    if len(common) >= min(10, max_frames):
        common_times = np.asarray(common, dtype=np.float64)
        selected = np.rint(
            np.linspace(0, len(common_times) - 1, min(max_frames, len(common_times)))
        ).astype(int)
        return common_times[np.unique(selected)]

    final_time = min(float(baseline_times[-1]), float(mitigated_times[-1]))
    return np.linspace(0.0, final_time, max_frames)


def create_mitigation_comparison_animation(
    baseline: FiniteRunResult,
    mitigated: FiniteRunResult,
    *,
    method: str,
    baseline_label: str,
    mitigated_label: str,
    baseline_case: str,
    mitigated_case: str,
    velocity: float,
    length: float,
    output: Path | str,
    max_frames: int = 90,
    fps: int = 20,
    dpi: int = 100,
    preview: bool = True,
) -> Path:
    """Create a synchronized side-by-side mitigation comparison video."""
    import matplotlib.pyplot as plt
    from matplotlib.animation import FFMpegWriter, FuncAnimation

    if max_frames < 2:
        raise ValueError("max_frames must be at least 2")
    if baseline.snapshot_times is None or mitigated.snapshot_times is None:
        raise ValueError("Both simulations must contain snapshots")

    ensure_ffmpeg()
    output = Path(output)
    if output.suffix.lower() != ".mp4":
        raise ValueError("The animation output filename must end in .mp4")
    output.parent.mkdir(parents=True, exist_ok=True)

    frame_times = _select_synchronized_frame_times(
        baseline, mitigated, max_frames
    )
    frame_count = len(frame_times)
    baseline_frames = _interpolate_snapshot_history(baseline, frame_times)
    mitigated_frames = _interpolate_snapshot_history(mitigated, frame_times)

    baseline_exact = np.stack([
        exact_finite_solution(
            baseline.x,
            float(time_now),
            velocity=velocity,
            length=length,
            case=baseline_case,
        )
        for time_now in frame_times
    ])
    mitigated_exact = np.stack([
        exact_finite_solution(
            mitigated.x,
            float(time_now),
            velocity=velocity,
            length=length,
            case=mitigated_case,
        )
        for time_now in frame_times
    ])
    baseline_norm = max(float(np.linalg.norm(baseline_frames[0])), 1.0e-30)
    mitigated_norm = max(float(np.linalg.norm(mitigated_frames[0])), 1.0e-30)

    all_values = np.concatenate([
        baseline_frames.ravel(),
        mitigated_frames.ravel(),
        baseline_exact.ravel(),
        mitigated_exact.ravel(),
    ])
    data_min = min(float(np.min(all_values)), 0.0)
    data_max = max(float(np.max(all_values)), 1.0)
    padding = max(0.08 * (data_max - data_min), 0.05)

    fig, axes = plt.subplots(1, 2, figsize=(11.4, 4.6), sharey=True)
    numerical_lines = []
    exact_lines = []
    time_texts = []
    error_texts = []
    for ax, label in zip(axes, (baseline_label, mitigated_label)):
        numerical_line, = ax.plot([], [], linewidth=2.0, label="Numerical")
        exact_line, = ax.plot([], [], "k--", linewidth=1.8, label="Exact")
        time_text = ax.text(0.02, 0.96, "", transform=ax.transAxes, va="top")
        error_text = ax.text(0.02, 0.88, "", transform=ax.transAxes, va="top")
        ax.set_xlim(0.0, length)
        ax.set_ylim(data_min - padding, data_max + padding)
        ax.set_xlabel("x")
        ax.set_title(label)
        ax.grid(True, alpha=0.3)
        ax.legend(loc="upper right")
        numerical_lines.append(numerical_line)
        exact_lines.append(exact_line)
        time_texts.append(time_text)
        error_texts.append(error_text)
    axes[0].set_ylabel("u")
    fig.suptitle(f"Mitigation study: {method}")
    fig.tight_layout()

    def initialize():
        artists = []
        for numerical_line, exact_line, time_text, error_text in zip(
            numerical_lines, exact_lines, time_texts, error_texts
        ):
            numerical_line.set_data([], [])
            exact_line.set_data([], [])
            time_text.set_text("")
            error_text.set_text("")
            artists.extend([numerical_line, exact_line, time_text, error_text])
        return tuple(artists)

    def update(frame: int):
        numerical_data = (baseline_frames[frame], mitigated_frames[frame])
        exact_data = (baseline_exact[frame], mitigated_exact[frame])
        grids = (baseline.x, mitigated.x)
        norms = (baseline_norm, mitigated_norm)
        artists = []
        for index in range(2):
            numerical_lines[index].set_data(grids[index], numerical_data[index])
            exact_lines[index].set_data(grids[index], exact_data[index])
            error = float(
                np.linalg.norm(numerical_data[index] - exact_data[index])
                / norms[index]
            )
            time_texts[index].set_text(f"t = {frame_times[frame]:.4f}")
            error_texts[index].set_text(f"relative L2 error = {error:.3e}")
            artists.extend([
                numerical_lines[index],
                exact_lines[index],
                time_texts[index],
                error_texts[index],
            ])
        return tuple(artists)

    animation = FuncAnimation(
        fig,
        update,
        frames=frame_count,
        init_func=initialize,
        interval=1000.0 / fps,
        blit=True,
    )
    writer = FFMpegWriter(
        fps=fps,
        bitrate=2400,
        metadata={"title": f"{method} mitigation comparison"},
    )
    animation.save(str(output), writer=writer, dpi=dpi)
    plt.close(fig)

    print(
        f"Video saved to {output.resolve()} "
        f"({frame_count} synchronized frames, {fps} fps)."
    )
    if preview:
        from IPython.display import Video, display as ipy_display

        ipy_display(Video(str(output), embed=True))
    return output.resolve()


def run_mitigation_comparison_example(
    method: str,
    *,
    baseline_parameters: dict[str, object],
    mitigated_parameters: dict[str, object],
    baseline_label: str,
    mitigated_label: str,
    output: Path | str,
    max_frames: int = 90,
    fps: int = 20,
    dpi: int = 100,
    preview: bool = True,
    download: bool = False,
) -> tuple[dict[str, FiniteRunResult], Path]:
    """Run two parameter choices and animate their synchronized comparison."""
    defaults: dict[str, object] = {
        "closure": "outgoing",
        "case": "cosine",
        "nx": 100,
        "velocity": 1.0,
        "cfl": 0.8,
        "final_time": 0.56,
        "length": 1.0,
        "startup": "lax-wendroff",
    }
    baseline_config = defaults | baseline_parameters
    mitigated_config = defaults | mitigated_parameters
    for name in ("velocity", "length"):
        if baseline_config[name] != mitigated_config[name]:
            raise ValueError(f"Both comparison runs must use the same {name}")

    def solve_config(config: dict[str, object]) -> FiniteRunResult:
        return solve_finite_interval(
            method,
            closure=str(config["closure"]),
            case=str(config["case"]),
            nx=int(config["nx"]),
            velocity=float(config["velocity"]),
            cfl=float(config["cfl"]),
            final_time=float(config["final_time"]),
            length=float(config["length"]),
            startup=str(config["startup"]),
            save_every=10**9,
            store_snapshots=True,
            snapshot_every=1,
        )

    baseline = solve_config(baseline_config)
    mitigated = solve_config(mitigated_config)
    print(
        f"{baseline_label}: final relative L2 error = "
        f"{baseline.relative_l2_error:.6e}"
    )
    print(
        f"{mitigated_label}: final relative L2 error = "
        f"{mitigated.relative_l2_error:.6e}"
    )
    video_path = create_mitigation_comparison_animation(
        baseline,
        mitigated,
        method=method,
        baseline_label=baseline_label,
        mitigated_label=mitigated_label,
        baseline_case=str(baseline_config["case"]),
        mitigated_case=str(mitigated_config["case"]),
        velocity=float(baseline_config["velocity"]),
        length=float(baseline_config["length"]),
        output=output,
        max_frames=max_frames,
        fps=fps,
        dpi=dpi,
        preview=preview,
    )
    if download:
        download_generated_file(video_path)
    return {"baseline": baseline, "mitigated": mitigated}, video_path


def run_validation_campaign(
    *,
    output_directory: Path | str = "advection_validation_videos",
    preview_each: bool = False,
    download_archive: bool = False,
    fps: int = 20,
    dpi: int = 100,
) -> tuple[dict[str, object], dict[str, Path], Path]:
    """Generate ten animated validation cases and package them in one ZIP.

    Cases 1--6 form the original method, closure, and instability campaign.
    Cases 7--10 are synchronized, side-by-side demonstrations of how grid
    refinement, the Courant number, and input smoothness mitigate numerical
    dissipation or dispersion.
    """
    import zipfile

    output_directory = Path(output_directory)
    output_directory.mkdir(parents=True, exist_ok=True)
    results: dict[str, object] = {}
    videos: dict[str, Path] = {}

    common = dict(
        closure="outgoing",
        case="cosine",
        nx=100,
        velocity=1.0,
        cfl=0.8,
        final_time=0.55,
        length=1.0,
        max_frames=90,
        fps=fps,
        dpi=dpi,
        preview=preview_each,
        download=False,
    )
    method_cases = (
        ("01_upwind_dissipation", "upwind"),
        ("02_lax_friedrichs_dissipation", "lax-friedrichs"),
        ("03_lax_wendroff_low_dissipation", "lax-wendroff"),
        ("04_leapfrog_low_dissipation", "leapfrog"),
    )
    for case_name, method in method_cases:
        print(f"\nGenerating {case_name}.mp4")
        result, video = run_animation_example(
            method=method,
            output=output_directory / f"{case_name}.mp4",
            **common,
        )
        results[case_name] = result
        videos[case_name] = video

    print("\nGenerating 05_leapfrog_closure_comparison.mp4")
    closure_results, closure_video = run_closure_animation_example(
        nx=120,
        velocity=1.0,
        cfl=0.8,
        final_time=1.10,
        length=1.0,
        case="cosine",
        max_frames=100,
        fps=fps,
        dpi=dpi,
        output=output_directory / "05_leapfrog_closure_comparison.mp4",
        preview=preview_each,
        download=False,
    )
    results["05_leapfrog_closure_comparison"] = closure_results
    videos["05_leapfrog_closure_comparison"] = closure_video

    print("\nGenerating 06_downwind_instability.mp4")
    downwind_result, downwind_video = run_animation_example(
        method="downwind",
        closure="outgoing",
        case="cosine",
        nx=80,
        velocity=1.0,
        cfl=0.4,
        final_time=0.10,
        length=1.0,
        max_frames=60,
        fps=max(10, fps // 2),
        dpi=dpi,
        output=output_directory / "06_downwind_instability.mp4",
        preview=preview_each,
        download=False,
        dynamic_ylim=True,
    )
    results["06_downwind_instability"] = downwind_result
    videos["06_downwind_instability"] = downwind_video


    print("\nGenerating 07_upwind_grid_refinement.mp4")
    upwind_refinement, upwind_refinement_video = run_mitigation_comparison_example(
        "upwind",
        baseline_parameters={"nx": 100, "cfl": 0.8, "final_time": 0.56},
        mitigated_parameters={"nx": 400, "cfl": 0.8, "final_time": 0.56},
        baseline_label=r"Baseline: $N_x=100$",
        mitigated_label=r"Refined: $N_x=400$",
        output=output_directory / "07_upwind_grid_refinement.mp4",
        max_frames=90,
        fps=fps,
        dpi=dpi,
        preview=preview_each,
        download=False,
    )
    results["07_upwind_grid_refinement"] = upwind_refinement
    videos["07_upwind_grid_refinement"] = upwind_refinement_video

    print("\nGenerating 08_lax_friedrichs_courant_mitigation.mp4")
    lf_courant, lf_courant_video = run_mitigation_comparison_example(
        "lax-friedrichs",
        baseline_parameters={"nx": 100, "cfl": 0.5, "final_time": 0.50},
        mitigated_parameters={"nx": 100, "cfl": 1.0, "final_time": 0.50},
        baseline_label=r"Diffusive: $C=0.5$",
        mitigated_label=r"Limiting case: $C=1$",
        output=output_directory / "08_lax_friedrichs_courant_mitigation.mp4",
        max_frames=90,
        fps=fps,
        dpi=dpi,
        preview=preview_each,
        download=False,
    )
    results["08_lax_friedrichs_courant_mitigation"] = lf_courant
    videos["08_lax_friedrichs_courant_mitigation"] = lf_courant_video

    print("\nGenerating 09_lax_wendroff_grid_refinement.mp4")
    lw_refinement, lw_refinement_video = run_mitigation_comparison_example(
        "lax-wendroff",
        baseline_parameters={"nx": 100, "cfl": 0.8, "final_time": 0.56},
        mitigated_parameters={"nx": 400, "cfl": 0.8, "final_time": 0.56},
        baseline_label=r"Baseline: $N_x=100$",
        mitigated_label=r"Refined: $N_x=400$",
        output=output_directory / "09_lax_wendroff_grid_refinement.mp4",
        max_frames=90,
        fps=fps,
        dpi=dpi,
        preview=preview_each,
        download=False,
    )
    results["09_lax_wendroff_grid_refinement"] = lw_refinement
    videos["09_lax_wendroff_grid_refinement"] = lw_refinement_video

    print("\nGenerating 10_leapfrog_smooth_initial_data.mp4")
    leapfrog_smoothness, leapfrog_smoothness_video = run_mitigation_comparison_example(
        "leapfrog",
        baseline_parameters={
            "nx": 100, "cfl": 0.8, "final_time": 0.56, "case": "cosine"
        },
        mitigated_parameters={
            "nx": 100, "cfl": 0.8, "final_time": 0.56, "case": "gaussian"
        },
        baseline_label="Compact cosine pulse",
        mitigated_label="Smooth Gaussian pulse",
        output=output_directory / "10_leapfrog_smooth_initial_data.mp4",
        max_frames=90,
        fps=fps,
        dpi=dpi,
        preview=preview_each,
        download=False,
    )
    results["10_leapfrog_smooth_initial_data"] = leapfrog_smoothness
    videos["10_leapfrog_smooth_initial_data"] = leapfrog_smoothness_video

    archive = output_directory.with_suffix(".zip").resolve()
    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
        for video in videos.values():
            bundle.write(video, arcname=video.name)
    print(f"\nValidation archive saved to {archive}")
    if download_archive:
        download_generated_file(archive)
    return results, videos, archive


## 10. Optional command-line interface

`main(argv)` remains available if the notebook is exported to a Python script.
Inside Jupyter, call `main([])` to use parser defaults, or preferably use one of
the example functions above. The notebook deliberately does **not** execute
`raise SystemExit(main())`: in an IPython kernel that statement reports the
successful exit code `0` as an exception and produces the warning shown in the
question.

In [11]:
def build_parser() -> argparse.ArgumentParser:
    """Build the command-line parser without reading process arguments."""
    parser = argparse.ArgumentParser(
        description="Solve 1D advection on a finite interval."
    )
    parser.add_argument(
        "--method",
        choices=[
            "upwind", "downwind", "lax-friedrichs", "lax-wendroff",
            "leapfrog", "ftcs"
        ],
        default="lax-wendroff",
    )
    parser.add_argument(
        "--outflow",
        choices=["frozen", "copy", "extrapolation", "outgoing"],
        default="outgoing",
    )
    parser.add_argument(
        "--startup",
        choices=["lax-wendroff", "upwind", "ftcs"],
        default="lax-wendroff",
    )
    parser.add_argument("--case", choices=["cosine", "gaussian"], default="cosine")
    parser.add_argument("--nx", type=int, default=400)
    parser.add_argument("--velocity", type=float, default=1.0)
    parser.add_argument("--cfl", type=float, default=0.8)
    parser.add_argument("--final-time", type=float, default=1.10)
    parser.add_argument("--length", type=float, default=1.0)
    parser.add_argument("--plot", type=Path)
    parser.add_argument("--closure-study", type=Path)
    return parser


def main(argv: Sequence[str] | None = None) -> int:
    """Command-line entry point; pass ``[]`` when calling it in Jupyter."""
    args = build_parser().parse_args(argv)

    if args.closure_study is not None:
        run_closure_example(output=args.closure_study, show=False)

    run_single_example(
        method=args.method,
        closure=args.outflow,
        startup=args.startup,
        case=args.case,
        nx=args.nx,
        velocity=args.velocity,
        cfl=args.cfl,
        final_time=args.final_time,
        length=args.length,
        output=args.plot,
        show=False,
    )
    return 0

## 11. Extended animated validation campaign

The first six videos retain the original validation structure:

| Video | Purpose | Expected observation |
|---|---|---|
| 1. Upwind | First-order reference | Amplitude loss and pulse broadening |
| 2. Lax--Friedrichs | Strongly diffusive reference | More diffusion than Upwind for the same $C<1$ |
| 3. Lax--Wendroff | Second-order method | Low dissipation with dispersive ripples |
| 4. Leapfrog | Second-order two-step method | Low dissipation and a dispersive computational component |
| 5. Leapfrog closures | Boundary validation | Different residuals and reflections after outflow crossing |
| 6. Downwind | Deliberately unstable stencil | Rapid growth of error and amplitude |

Cases 1--4 use the same compact cosine pulse, grid, Courant number, and final
time and stop before the pulse reaches the outflow boundary. Case 5 extends
the run beyond boundary crossing. Case 6 is shorter because its instability
grows rapidly.

### Mitigation experiments

Four additional videos show that the preceding numerical artifacts can be
reduced. Each video contains synchronized baseline and mitigated panels, so
the improvement can be judged at the same physical time.

| Video | Parameter change | What it demonstrates |
|---|---|---|
| 7. Upwind refinement | $N_x:100\rightarrow400$, fixed $C=0.8$ | Numerical diffusion decreases with $\Delta x$ |
| 8. Lax--Friedrichs CFL | $C:0.5\rightarrow1$ at fixed grid | Its artificial diffusion decreases as $C$ approaches one |
| 9. Lax--Wendroff refinement | $N_x:100\rightarrow400$, fixed $C=0.8$ | Dispersive deformation decreases under refinement |
| 10. Leapfrog smooth data | compact cosine $\rightarrow$ Gaussian | Reduced high-wavenumber content produces less ringing |

The $C=1$ Lax--Friedrichs result is a special limiting case for constant
positive velocity on a uniform grid: its update becomes the exact one-cell
shift $U_j^{n+1}=U_{j-1}^n$. It should not be interpreted as a general license
to choose the time step solely to suppress diffusion.

The Gaussian comparison changes the input rather than the discretization. It
shows how solution regularity affects a dispersive scheme; it does not claim
that smoothing the physical initial condition is always admissible.

The ten videos remain only a selected campaign. The implementation also
supports other grids, Courant numbers, final times, domain lengths, positive
velocities, startup methods, individual closures, and the FTCS counterexample.

### Generate and download all ten videos

The following cell creates ten MP4 files and packages them in one ZIP archive.
In Google Colab, `download_archive=True` opens one download dialog. Set
`preview_each=True` to embed all ten videos; leaving it `False` avoids a very
large notebook output.

In [12]:
campaign_results, campaign_videos, campaign_archive = run_validation_campaign(
    output_directory="advection_validation_videos",
    preview_each=False,
    download_archive=True,
    fps=20,
    dpi=100,
)


Generating 01_upwind_dissipation.mp4
method                  = upwind
outflow closure         = outgoing
case                    = cosine
grid cells              = 100
time steps              = 69
dt                      = 7.971014492754e-03
Courant number          = 7.971014492754e-01
relative L2 error       = 2.193796767955e-01
residual L2 ratio       = 8.689656934247e-01
maximum residual        = 7.880046537124e-01
Video saved to /content/advection_validation_videos/01_upwind_dissipation.mp4 (70 frames, 20 fps).

Generating 02_lax_friedrichs_dissipation.mp4
method                  = lax-friedrichs
outflow closure         = outgoing
case                    = cosine
grid cells              = 100
time steps              = 69
dt                      = 7.971014492754e-03
Courant number          = 7.971014492754e-01
relative L2 error       = 3.701766152792e-01
residual L2 ratio       = 7.788703640948e-01
maximum residual        = 6.378697199818e-01
Video saved to /content/advection_valid

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Run a selected mitigation animation

These are alternatives to the complete campaign and are therefore commented.

In [13]:
# Upwind, coarse versus refined grid:
# results_07, video_07 = run_mitigation_comparison_example(
#     "upwind",
#     baseline_parameters={"nx": 100, "cfl": 0.8, "final_time": 0.56},
#     mitigated_parameters={"nx": 400, "cfl": 0.8, "final_time": 0.56},
#     baseline_label=r"Baseline: $N_x=100$",
#     mitigated_label=r"Refined: $N_x=400$",
#     output="07_upwind_grid_refinement.mp4",
#     preview=True,
#     download=True,
# )

# Leapfrog, compact cosine versus smooth Gaussian pulse:
# results_10, video_10 = run_mitigation_comparison_example(
#     "leapfrog",
#     baseline_parameters={"case": "cosine"},
#     mitigated_parameters={"case": "gaussian"},
#     baseline_label="Compact cosine pulse",
#     mitigated_label="Smooth Gaussian pulse",
#     output="10_leapfrog_smooth_initial_data.mp4",
#     preview=True,
#     download=True,
# )

To download the archive again without recomputing or re-encoding the videos:

In [14]:
# download_generated_file("advection_validation_videos.zip")